## Executive Summary

This notebook computes **winding factors** — the efficiency with which a winding produces each spatial harmonic of the airgap field.

| Function | What it computes |
|----------|------------------|
| `pitch_factor` | kp(ν) — chording factor |
| `distribution_factor` | kd(ν) — belt factor (integer-slot only) |
| `winding_factor` | kw(ν) = kp·kd or phasor sum |
| `winding_factor_spectrum` | |kw(ν)| for ν = 1…ν_max |

## How It Fits Into Motor Design

```
01_winding_sos.ipynb  (layout engine)
      │
      ▼
01_winding_factors.ipynb   ←── YOU ARE HERE
      │
      ▼
01_winding_mmf.ipynb       (MMF waveform & spectrum)
```

**Depends on:** `sos.py` — uses `build_coil_matrix`, `get_valid_coil_spans`, `winding_factor_sos`.

**Feeds into:** `01_winding_mmf.ipynb` (via `winding_factor_spectrum` and `_optimal_coil_span`).

## Notation

| Symbol | Definition |
|--------|------------|
| Q | Stator slots |
| P | Poles |
| p | Pole pairs = P/2 |
| τp | Pole pitch = Q/P (one magnetic pole) |
| w | Coil span in slots |
| ν | Mechanical harmonic order |
| kp | Pitch (chording) factor |
| kd | Distribution (belt) factor |
| kw | Combined winding factor = kp·kd (ISW) |
| q | Slots per pole per phase = Q/(2mp) |

## Imports and Setup

In [ ]:
#| default_exp winding.winding_factors

In [ ]:
#| export
from __future__ import annotations

import numpy as np

from emachines.winding.sos import build_coil_matrix, get_valid_coil_spans, winding_factor_sos

__all__ = [
    "pitch_factor",
    "distribution_factor",
    "winding_factor",
    "winding_factor_spectrum",
]

---

## `pitch_factor`: Chording Factor kp

### Theory

Shortening a coil below the full pole pitch (chording) reduces harmonic content:

$$k_{p\nu} = \left|\sin\!\left(\nu \cdot \frac{\pi}{2} \cdot \frac{w}{\tau_p}\right)\right|$$

**Note on pole pitch:** τp = Q/P is one *magnetic* pole (half the electrical period). For 12s/4p: τp = 3 slots, full-pitch coil span = 3.

**References:** Hanselman (2003), eq. 4.5; Pyrhönen et al. (2008), §2.4.

In [ ]:
#| export
def pitch_factor(nu: int, coil_span: int, pole_pitch: float) -> float:
    """
    Pitch (chording) factor kp for the nu-th mechanical harmonic.

    kp(nu) = |sin(nu * pi/2 * w / tau_p)|

    Parameters
    ----------
    nu         : int    Harmonic order (1 = fundamental)
    coil_span  : int    Coil span in slots (w)
    pole_pitch : float  Pole pitch in slots (tau_p = Q/P, one magnetic pole)

    Returns
    -------
    float   kp in [0, 1]

    Examples
    --------
    >>> pitch_factor(1, 6, 6.0)   # full-pitch
    1.0
    >>> round(pitch_factor(1, 5, 6.0), 4)   # 5/6 chording
    0.9659
    """
    return float(np.abs(np.sin(nu * np.pi / 2 * coil_span / pole_pitch)))

In [ ]:
from emachines.winding.winding_factors import pitch_factor

# 12s/4p: pole_pitch = Q/P = 3
pole_pitch = 3.0
print("Pitch factor for 12s/4p (tau_p = 3 slots):")
print(f"{'w':>4}  {'kp(1)':>8}  {'kp(3)':>8}  {'kp(5)':>8}")
print("-" * 35)
for w in range(1, 7):
    kp1 = pitch_factor(1, w, pole_pitch)
    kp3 = pitch_factor(3, w, pole_pitch)
    kp5 = pitch_factor(5, w, pole_pitch)
    marker = " <- full-pitch" if w == int(pole_pitch) else ""
    print(f"  {w:>2}  {kp1:>8.4f}  {kp3:>8.4f}  {kp5:>8.4f}{marker}")

---

## `distribution_factor`: Belt Factor kd

### Theory

Spreading conductors across q slots per pole per phase:

$$k_{d\nu} = \frac{\sin(\nu \pi / (2m))}{q \cdot \sin(\nu \pi / (2mq))}$$

where q = Q/(2mp). Valid for integer-slot windings (q ≥ 1) only.

**References:** Pyrhönen et al. (2008), eq. 2.15.

In [ ]:
#| export
def distribution_factor(nu: int, Q: int, p: int, m: int = 3) -> float:
    """
    Distribution (belt) factor kd for the nu-th harmonic.

    Valid for integer-slot windings (q >= 1).

    kd(nu) = sin(nu*pi/(2m)) / (q * sin(nu*pi/(2mq)))

    Parameters
    ----------
    nu : int   Harmonic order
    Q  : int   Total number of slots
    p  : int   Number of pole pairs
    m  : int   Number of phases (default 3)

    Returns
    -------
    float   kd in (0, 1]

    Raises
    ------
    ValueError
        If q < 1. Use winding_factor() for FSCW.

    Examples
    --------
    >>> round(distribution_factor(1, 12, 2), 4)   # q=1
    1.0
    >>> round(distribution_factor(1, 24, 2), 4)   # q=2
    0.9659
    """
    q = Q / (2 * m * p)
    if q < 1.0 - 1e-9:
        raise ValueError(
            f"distribution_factor() requires q >= 1 (integer-slot winding). "
            f"Got q = {q:.4f} (Q={Q}, p={p}, m={m}). "
            f"Use winding_factor() for FSCW."
        )
    num = np.sin(nu * np.pi / (2 * m))
    den = q * np.sin(nu * np.pi / (2 * m * q))
    if np.abs(den) < 1e-12:
        return 1.0
    return float(np.abs(num / den))

In [ ]:
from emachines.winding.winding_factors import distribution_factor
import fractions

print("Distribution factor kd(nu=1) vs q:")
print(f"{'Config':>20}  {'q':>6}  {'kd(1)':>8}")
print("-" * 40)
for Q, p, label in [(12,2,"12s/4p q=1"), (24,2,"24s/4p q=2"), (36,2,"36s/4p q=3")]:
    q = fractions.Fraction(Q, 2*3*p)
    kd = distribution_factor(1, Q, p)
    print(f"  {label:>18}  {str(q):>6}  {kd:>8.4f}")

---

## `winding_factor`: Combined kw

### Theory

kw(ν) = kp(ν) · kd(ν) for integer-slot (q ≥ 1), or star-of-slots phasor sum for FSCW (q < 1).

**Coil span note:** `coil_span` here is the pole pitch τp = Q/P (one magnetic pole). For 12s/4p: τp = 3, so `coil_span=3` for full-pitch. Compare: `build_coil_matrix` uses the full electrical period Q//p = 6 as its 'full_pitch' — a different quantity.

In [ ]:
#| export
def winding_factor(
    nu: int,
    Q: int,
    p: int,
    coil_span: int,
    m: int = 3,
) -> float:
    """
    Combined winding factor kw for harmonic nu.

    Dispatches automatically:
    - Integer-slot (q >= 1): kw = kp * kd
    - FSCW (q < 1): kw via star-of-slots phasor sum

    Parameters
    ----------
    nu        : int   Harmonic order (1 = fundamental)
    Q         : int   Total number of slots
    p         : int   Number of pole pairs
    coil_span : int   Coil span in slots.
                      Full-pitch for ISW = Q/P (one magnetic pole).
                      Tooth-coil for FSCW = 1.
    m         : int   Number of phases (default 3)

    Returns
    -------
    float   kw in [0, 1]

    Examples
    --------
    >>> winding_factor(1, 12, 2, coil_span=3)   # 12s/4p full-pitch (tau_p=3)
    1.0
    >>> round(winding_factor(5, 12, 5, coil_span=1), 4)   # 12s/10p FSCW working harmonic (nu=p=5)
    0.933
    >>> round(winding_factor(4, 12, 4, coil_span=1), 4)   # 12s/8p FSCW working harmonic (nu=p=4)
    0.866
    """
    P = 2 * p
    pole_pitch = Q / P
    q = Q / (2 * m * p)

    if q >= 1.0 - 1e-9:
        kp = pitch_factor(nu, coil_span, pole_pitch)
        kd = distribution_factor(nu, Q, p, m)
        return kp * kd
    else:
        return winding_factor_sos(nu, Q, P, m, layers=2, w=coil_span)

In [ ]:
from emachines.winding.winding_factors import winding_factor

configs = [
    dict(Q=12, p=2, coil_span=3, label="12s/4p  ISW full-pitch"),
    dict(Q=24, p=2, coil_span=6, label="24s/4p  ISW full-pitch"),
    dict(Q=12, p=5, coil_span=1, label="12s/10p FSCW tooth-coil"),
    dict(Q=12, p=4, coil_span=1, label="12s/8p  FSCW tooth-coil"),
]
print(f"{'Configuration':<30}  {'kw(nu=1)':>9}  {'kw(nu=p)':>9}")
print("-" * 55)
for c in configs:
    kw1 = winding_factor(1, c['Q'], c['p'], c['coil_span'])
    kwp = winding_factor(c['p'], c['Q'], c['p'], c['coil_span'])
    print(f"  {c['label']:<28}  {kw1:>9.4f}  {kwp:>9.4f}")

---

## `_optimal_coil_span` (private)

Finds the coil span that maximises |kw| at the working harmonic ν = p. Used internally when `w=None` in `winding_factor_spectrum` and `mmf.py`.

In [ ]:
#| export
def _optimal_coil_span(Q: int, P: int, m: int = 3, layers: int = 1) -> int:
    """
    Coil span (slots) that maximises kw at the working harmonic.

    Parameters
    ----------
    Q, P, m : int
    layers  : int

    Returns
    -------
    int

    Examples
    --------
    >>> _optimal_coil_span(12, 10)
    1
    >>> _optimal_coil_span(24, 4)
    6
    """
    p = P // 2
    q = Q / (2 * m * p)
    # For integer-slot windings, full pitch (Q//P) always maximises kp=1.
    # The SOS spectrum cannot distinguish spans in SL-ISW (only DL layers differ),
    # so we short-circuit here.
    if q >= 1.0 - 1e-9:
        return max(1, Q // P)
    spans = get_valid_coil_spans(Q, P)
    best_w, best_kw = spans[0], -1.0
    for span in spans:
        try:
            _, kw = winding_factor_spectrum(Q, P, m=m, layers=layers, w=span, nu_max=p)
            kw1 = float(kw[p - 1]) if p <= len(kw) else 0.0
            if kw1 > best_kw:
                best_kw, best_w = kw1, span
        except Exception:
            pass
    return best_w

In [ ]:
from emachines.winding.winding_factors import _optimal_coil_span

for Q, P in [(12, 10), (12, 8), (9, 8), (24, 4), (36, 4)]:
    w = _optimal_coil_span(Q, P)
    print(f"  {Q}s/{P}p  optimal w = {w}  (full_pitch = {Q//(P//2)})")

---

## `winding_factor_spectrum`: Full Harmonic Spectrum

### Theory

The phasor sum method computes |kw(ν)| for all mechanical harmonics:

$$k_{w\nu} = \frac{1}{m} \sum_{k=0}^{m-1} \frac{\left|\sum_{i,\text{lyr}} s_{i,\text{lyr}} e^{j\nu 2\pi i/Q}\right|}{N_k}$$

Works for all winding types. **Working harmonic:** ν = p = P/2.

**References:** Müller, Vogt & Ponick (2008), eq. 3.65.

In [ ]:
#| export
def winding_factor_spectrum(
    Q: int,
    P: int,
    m: int = 3,
    layers: int = 1,
    w: int | None = None,
    nu_max: int = 40,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Winding factor magnitude spectrum for mechanical harmonics 1...nu_max.

    Parameters
    ----------
    Q      : int   Number of stator slots
    P      : int   Number of poles (even, >= 2)
    m      : int   Number of phases (default 3)
    layers : int   1 (single-layer) or 2 (double-layer)
    w      : int | None
             Coil span in slots. None -> span that maximises kw.
    nu_max : int   Highest mechanical harmonic (default 40)

    Returns
    -------
    nu_arr : np.ndarray, shape (nu_max,), dtype int
    kw_arr : np.ndarray, shape (nu_max,), dtype float64
        |kw(nu)| in [0, 1]

    Examples
    --------
    >>> nu, kw = winding_factor_spectrum(12, 10, layers=2)
    >>> round(float(kw[4]), 4)   # nu=5 working harmonic for 12s/10p
    0.933
    >>> nu, kw = winding_factor_spectrum(12, 4, layers=1, w=6, nu_max=10)
    >>> round(float(kw[1]), 4)   # nu=2, 12s/4p q=1 full-pitch
    1.0
    """
    if w is None:
        w = _optimal_coil_span(Q, P, m, layers)

    matrix = build_coil_matrix(Q, P, m, layers, w)
    slots = np.arange(Q, dtype=np.float64)
    nu_arr = np.arange(1, nu_max + 1, dtype=int)
    kw_arr = np.zeros(len(nu_arr), dtype=np.float64)

    for idx, nu in enumerate(nu_arr):
        phasors = np.exp(1j * nu * 2.0 * np.pi * slots / Q)
        kw_vals = []
        for k in range(m):
            ph = k + 1
            total = 0j
            n_cond = 0
            for lyr in range(layers):
                fwd = matrix[lyr] == +ph
                ret = matrix[lyr] == -ph
                total += phasors[fwd].sum() - phasors[ret].sum()
                n_cond += int(fwd.sum()) + int(ret.sum())
            if n_cond > 0:
                kw_vals.append(abs(total) / n_cond)
        kw_arr[idx] = float(np.mean(kw_vals)) if kw_vals else 0.0

    return nu_arr, kw_arr

In [ ]:
from emachines.winding.winding_factors import winding_factor_spectrum
import numpy as np

configs = [
    dict(Q=12, P=10, layers=2, label="12s/10p FSCW DL"),
    dict(Q=12, P=8,  layers=2, label="12s/8p  FSCW DL"),
    dict(Q=24, P=4,  layers=2, label="24s/4p  ISW  DL"),
]

for cfg in configs:
    nu, kw = winding_factor_spectrum(cfg['Q'], cfg['P'], layers=cfg['layers'], nu_max=15)
    p = cfg['P'] // 2
    print(f"\n{cfg['label']}  (working nu = {p})")
    for i in range(len(nu)):
        if kw[i] > 0.01:
            marker = " <- working" if nu[i] == p else ""
            print(f"  nu={int(nu[i]):>3}  kw={float(kw[i]):>6.4f}{marker}")

---

## References

1. Hanselman, D.C. (2003). *Brushless Permanent Magnet Motor Design*, 2nd ed. eq. 4.5.
2. Pyrhönen, J., Jokinen, T., & Hrabovcová, V. (2008). *Design of Rotating Electrical Machines*. Wiley. eq. 2.15.
3. Müller, G., Vogt, K., & Ponick, B. (2008). *Berechnung elektrischer Maschinen*. Wiley-VCH. eq. 3.65.
4. Bianchi, N., & Bolognani, S. (2002). IEEE Trans. Ind. Appl., 38(5). DOI: 10.1109/TIA.2002.802909.

---

## Tests

In [ ]:
#| hide
import math
import numpy as np
from emachines.winding.winding_factors import (
    pitch_factor, distribution_factor, winding_factor,
    winding_factor_spectrum, _optimal_coil_span,
)

# pitch_factor
assert math.isclose(pitch_factor(1, 6, 6.0), 1.0), "full-pitch kp=1"
assert math.isclose(pitch_factor(1, 5, 6.0), 0.9659, abs_tol=1e-4), "5/6 chording"
assert math.isclose(pitch_factor(3, 4, 6.0), 0.0, abs_tol=1e-10), "3rd harmonic cancelled"
print("pass pitch_factor")

# distribution_factor
assert math.isclose(distribution_factor(1, 12, 2), 1.0, abs_tol=1e-10), "q=1 kd=1"
assert math.isclose(distribution_factor(1, 24, 2), 0.9659, abs_tol=1e-4), "q=2"
try:
    distribution_factor(1, 12, 5)
    assert False, "should raise"
except ValueError:
    pass
print("pass distribution_factor")

# winding_factor
assert math.isclose(winding_factor(1, 12, 2, coil_span=3), 1.0, abs_tol=1e-10)
assert math.isclose(winding_factor(5, 12, 5, coil_span=1), 0.9330, abs_tol=1e-3)  # nu=p=5 working harmonic
assert math.isclose(winding_factor(4, 12, 4, coil_span=1), 0.8660, abs_tol=1e-3)  # nu=p=4 working harmonic
print("pass winding_factor")

# _optimal_coil_span
assert _optimal_coil_span(12, 10) == 1
assert _optimal_coil_span(24, 4) == 6
print("pass _optimal_coil_span")

# winding_factor_spectrum
nu, kw = winding_factor_spectrum(12, 10, layers=2, nu_max=20)
assert nu.shape == (20,) and kw.shape == (20,)
assert math.isclose(float(kw[4]), 0.9330, abs_tol=1e-3)
assert np.all(kw >= -1e-10) and np.all(kw <= 1.0 + 1e-10)

_, kw8 = winding_factor_spectrum(12, 8, layers=2, nu_max=10)
assert math.isclose(float(kw8[3]), 0.8660, abs_tol=1e-3)

_, kw12 = winding_factor_spectrum(12, 4, layers=1, w=6, nu_max=10)
assert math.isclose(float(kw12[1]), 1.0, abs_tol=1e-4)
print("pass winding_factor_spectrum")

print()
print("All winding factor tests passed")

# ── Parameterised spectrum checks ─────────────────────────────────────────────
_spectrum_combos = [
    # (Q,  P, layers, nu_working, kw_expected, tol, label)
    (12, 10, 2, 5, 0.9330, 1e-3, '12s/10p FSCW'),
    (12, 14, 2, 7, 0.9330, 1e-3, '12s/14p FSCW'),
    (12,  8, 2, 4, 0.8660, 1e-3, '12s/8p  FSCW'),
    (24,  4, 2, 2, 0.9659, 1e-3, '24s/4p  ISW'),
    (36,  8, 2, 4, 0.9452, 1e-3, '36s/8p  ISW'),
]

for Q, P, layers, nu_w, kw_exp, tol, label in _spectrum_combos:
    w_opt = _optimal_coil_span(Q, P, m=3, layers=layers)
    nu_arr, kw_arr = winding_factor_spectrum(Q, P, layers=layers, w=w_opt, nu_max=nu_w*2)
    kw_val = float(kw_arr[nu_w - 1])
    assert math.isclose(kw_val, kw_exp, abs_tol=tol),         f"{label}: kw(nu={nu_w})={kw_val:.4f}, expected {kw_exp}"
    assert np.all(kw_arr >= -1e-10) and np.all(kw_arr <= 1.0 + 1e-10),         f"{label}: kw out of [0,1]"
    print(f"pass  {label}: kw(nu={nu_w})={kw_val:.4f}")
